# Heat Unit — Remaining 6 Clips (Wan2.1, Kaggle)

3 of the 9 Heat-unit beats (`sources`, `thermal_equilibrium`, `thermal_expansion`) were already
generated with paid Veo 3.1 clips and are being assembled locally. This notebook generates only
the other **6 beats** with the free, open-source Wan2.1-T2V-1.3B model, so you don't re-pay for
or duplicate the ones already done.

**When finished:** download the 6 files under `/kaggle/working/clips/` from the **Output** tab
(right sidebar) and place them in the local pipeline's `clips/` folder, named exactly:
`intro.mp4`, `what_is_heat.mp4`, `hot_cold_temperature.mp4`, `heat_vs_temperature.mp4`,
`uses_of_expansion.mp4`, `outro.mp4`.

**Before running:** Settings (right sidebar) > Accelerator > `GPU T4 x2` or `GPU P100`, and
Internet > On.


## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Settings (right sidebar) > Accelerator > 'GPU T4 x2' or 'GPU P100'. "
        "If already set, you may have hit Kaggle's weekly ~30 free GPU-hours quota."
    )
print("GPU OK:", torch.cuda.get_device_name(0))


In [ ]:
!pip install -q -U diffusers transformers accelerate bitsandbytes imageio imageio-ffmpeg ftfy


## 2. Load the open-source text-to-video model

`Wan-AI/Wan2.1-T2V-1.3B-Diffusers` (Apache-2.0), 4-bit text encoder to fit free-tier RAM.


In [ ]:
import os
import gc

import torch
from transformers import UMT5EncoderModel, BitsAndBytesConfig
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
WAN_W, WAN_H = 640, 368
WAN_FPS = 16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
)
text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder", quantization_config=quant_config, torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(
    MODEL_ID, vae=vae, text_encoder=text_encoder, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
gc.collect()
torch.cuda.empty_cache()

NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, extra limbs, bad anatomy, "
    "static image, watermark, text, subtitles, worst quality, jpeg artifacts, ugly"
)
print("model loaded:", MODEL_ID)


## 3. Generate the 6 remaining clips

Only these 6 — `sources`, `thermal_equilibrium`, `thermal_expansion` are already done via Veo.


In [ ]:
BEATS = [
    {"id": "intro",
     "prompt": "a bright sun radiating warm glowing heat waves over a green landscape, colorful educational cartoon animation style",
     "frames": 25},
    {"id": "what_is_heat",
     "prompt": "glowing molecules inside a solid cube vibrating and moving faster as heat energy is added, simple educational science animation, colorful",
     "frames": 21},
    {"id": "hot_cold_temperature",
     "prompt": "a cartoon thermometer dipped into a glass of hot water and a glass of cold water, red mercury rising and falling, colorful educational animation style",
     "frames": 21},
    {"id": "heat_vs_temperature",
     "prompt": "side by side comparison of a small steaming cup of tea and a large calm pond, educational infographic cartoon animation style, colorful",
     "frames": 21},
    {"id": "uses_of_expansion",
     "prompt": "a railway track with small visible gaps between the rails under bright sunshine, simple educational cartoon animation style, engineering diagram feel",
     "frames": 21},
    {"id": "outro",
     "prompt": "a happy cartoon classroom scene with a smiling student surrounded by icons of the sun, a thermometer, and a railway track, colorful warm animation style",
     "frames": 25},
]
print(len(BEATS), "beats to generate")


In [ ]:
os.makedirs("/kaggle/working/clips", exist_ok=True)


def generate_with_retry(prompt, num_frames, width=WAN_W, height=WAN_H, steps=30, attempt=0):
    try:
        output = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
            height=height, width=width, num_frames=num_frames,
            guidance_scale=5.0, num_inference_steps=steps,
        )
        return output.frames[0]
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        if attempt >= 1:
            raise
        half_w = max(16, (width // 2) // 16 * 16)
        half_h = max(16, (height // 2) // 16 * 16)
        print(f"  OOM at {width}x{height} — retrying once at {half_w}x{half_h}...")
        return generate_with_retry(prompt, num_frames, half_w, half_h, steps, attempt + 1)


for beat in BEATS:
    out_path = f"/kaggle/working/clips/{beat['id']}.mp4"
    if os.path.exists(out_path):
        print("skip (already exists):", beat["id"])
        continue
    print("generating:", beat["id"], "->", beat["prompt"])
    video_frames = generate_with_retry(beat["prompt"], beat["frames"])
    export_to_video(video_frames, out_path, fps=WAN_FPS)
    print("saved:", out_path)
    del video_frames
    gc.collect()
    torch.cuda.empty_cache()

print("all done — check the Output tab (right sidebar) to download the 6 files under clips/")
